In [ ]:
%run _bootstrap_dev.ipynb

## §2 — Portafoglio da analizzare

In [ ]:
#
# Quale portafoglio analizzare
#
# portfolio = portfolio_alpha_euro
# portfolio = portfolio_alpha_sect
# portfolio = portfolio_alpha_world
# portfolio = portfolio_alpha_world_vanguard
# portfolio = portfolio_alpha_nasdaq100
# portfolio = portfolio_alpha_sp100
portfolio = portfolio_germany_plan
# portfolio = portfolio_italy_big_cap
# portfolio = portfolio_alpha_quant # New Ema
# portfolio = portfolio_alpha_fact

# Profile di destinazione del PTF nel portafoglio del cliente
# Determina le soglie di overfitting check (S1-S4).
# "satellite": quota tattica, cerca alpha; "core": quota di base, prioritizza capital preservation.
profile = "satellite"   # "satellite" | "core"
pf_rot_cluster = None 
pf_rot_cluster_base = None

# --- Derivazione automatica dalle proprieta' del portfolio ---
# Flag calcolato PRIMA della risoluzione Wikipedia, quando tickers è ancora stringa.
survivorship_bias_universe = isinstance(portfolio['tickers'], str)
tickers = portfolio['tickers']
tickers = (
    extract_tickers_from_wikipedia(tickers, exclude=["GOOG"], rename={"BRK.B": "BRK-B"})
    if isinstance(tickers, str)
    else list(tickers)
)
benchmark_portfolio = portfolio['benchmark_portfolio']
benchmark_title     = portfolio['benchmark_title']
portfolio_title     = portfolio['Title']

# Percorso file WFO
wfo_results_dir = _TSLAB_DEV_R_WFO_RESULTS_DIR
wfo_file_save   = f"{wfo_results_dir}/{portfolio_title}_{year}.wfo_summary.csv"

# Date di analisi
start_date = "2015-01-01"
end_date   = None          # None = oggi
year       = 2026          # anno di selezione corrente

print(f"Portafoglio: {BOLD}{portfolio_title}{RESET}  |  Profile: {BOLD}{profile}{RESET}")
print(f"Ticker: {len(tickers)}  |  Benchmark: {benchmark_title}")
print(f"WFO file: {wfo_file_save}")
print(f"Survivorship bias universe: {survivorship_bias_universe}")

reports_dir = get_analysis_output_dir("r_analysis", ptf_name=portfolio_title.replace(' ', '_').lower())
plots_dir   = reports_dir / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)
print(f"reports_dir: {reports_dir}")
print(f"plots_dir: {plots_dir}")

## §3 — Download data

In [ ]:
init_cash      = 100_000
normalize      = False    # lasciare False nei rotazionali: NaN = titolo assente
lookback_buffer = 365

download_start_date = (pd.to_datetime(start_date) - timedelta(days=lookback_buffer)).strftime("%Y-%m-%d")

stocks_data, company_data = fetch_data_and_companies(
    tickers, download_start_date, end_date, normalize=normalize
)
stocks_data_raw = download_data(tickers, download_start_date, end_date, auto_adjust=False)

# Usato dalla Stability Analysis 
portfolio["stocks_data"] = stocks_data
portfolio["init_cash"]   = init_cash

if benchmark_portfolio:
    benchmark_data     = build_benchmark(benchmark_portfolio,
                             stocks_data.index.min(), stocks_data.index.max()).replace(0, np.nan).ffill()
    benchmark_data_raw = build_benchmark(benchmark_portfolio,
                             stocks_data.index.min(), stocks_data.index.max(),
                             auto_adjust=False).replace(0, np.nan).ffill()
elif benchmark_title:
    benchmark_data     = download_data(benchmark_title, stocks_data.index.min(), end_date)
    benchmark_data_raw = download_data(benchmark_title, stocks_data_raw.index.min(), end_date,
                                       auto_adjust=False)
else:
    benchmark_data = benchmark_data_raw = None

display(stocks_data)



In [ ]:
risk_off_tickers = portfolio.get('risk_off_tickers', risk_off_tickers)
risk_off_tickers_uniq = [
    t for t in risk_off_tickers
    if t not in tickers
]

# Tickers da usare in modalita risk_off (difensivi)
risk_off_data = download_data(risk_off_tickers_uniq, download_start_date, end_date) if risk_off_tickers else None
# workaround: forza sempre DataFrame
if isinstance(risk_off_data, pd.Series):
    risk_off_data = risk_off_data.to_frame()

## §4 Run WFO's with Stability Analysis

In [ ]:
# # Griglia completa — definita qui per essere condivisa tra stability e WFO
# full_grid = {
#     "rebalance_frequency":    ["QE", "ME"],
#     "momentum_lookback_days": [10, 20, 40, 60],
#     "riskparity_lookback_days": [10, 20, 40, 60],
#     "n_top":                  resolve_n_top(portfolio.get('asset_type', 'stock'), profile),
#     "use_acceleration":       [True, False],
#     "momentum_weight":        [0.5, 0.7, 1.0],
#     "filter_ema":             [True, False],
#     "filter_volatility":      [True, False],
#     "filter_min_momentum":    [True, False],
# }

# # Parametri WFO (condivisi tra Standard e pipeline interna Cluster)
# ratio                  = "3:1"
# metric                 = "Sharpe Ratio"
# cores                  = -1
# verbose                = False
# force_next_year_params = True

# n_full_trials = len(list(__import__('itertools').product(*full_grid.values())))
# print(f"Full grid size: {n_full_trials} combinazioni")
# print(f"n_top: {full_grid['n_top']}  (asset_type={portfolio.get('asset_type','stock')}, profile={profile})")



In [ ]:
# §4+§5a — Griglia WFO e pipeline unificata (Momentum + Multifactor)

# Parametri WFO (condivisi tra engines)
ratio                  = "3:1"
metric                 = "Sharpe Ratio"
cores                  = -1
verbose                = False
force_next_year_params = True


# build_wfo_grid() restituisce combinazioni pre-espanse; autoreduce=True
# chiama reduce_grid_via_stability internamente per entrambi gli engine.

ratio_int = int(str(ratio).split(':')[0])
benchmark_start    = benchmark_data.dropna(how='all').index.min()
first_full_year    = pd.Timestamp(f"{benchmark_start.year + 1}-01-01")
pipeline_start_date = first_full_year - pd.DateOffset(years=ratio_int)
autoreduce=True
risk_on_off=True
plot=True

print(f"Benchmark start:         {benchmark_start.date()}")
print(f"Primo anno pieno comune: {first_full_year.date()}")
print(f"Pipeline start:          {pipeline_start_date.date()}")
print(f"Primo OOS atteso:        {first_full_year.date()}")

results_pipeline = {}
for engine in ["Momentum", "Multifactor"]:
    
    grid = build_wfo_grid(engine=engine, profile=profile, asset_type=portfolio.get('asset_type', 'stock'))
    asset_type=portfolio.get('asset_type', 'stock')
    wfo_audit_path_base=str(reports_dir / f"{portfolio_title}_{year}_{engine.lower()}")

    results_pipeline[engine] = run_wfo_pipeline(
        stocks_data_raw=stocks_data_raw,
        stocks_data=stocks_data,
        benchmark_data=benchmark_data,
        benchmark_data_raw=benchmark_data_raw,
        tickers=tickers,
        risk_off_data=risk_off_data,
        ratio=ratio,
        metric=metric,
        start_date=pipeline_start_date,
        end_date=end_date,
        cores=cores,
        verbose=verbose,
        force_next_year_params=False,
        param_grid=grid,
        engine=engine,
        autoreduce=autoreduce,
        portfolio_title=portfolio_title,
        benchmark_title=benchmark_title,
        init_cash=init_cash,
        risk_on_off=risk_on_off,
        plot=plot,
        profile=profile,
        asset_type=asset_type,
        wfo_audit_path_base=wfo_audit_path_base,
        benchmark_prices=benchmark_data_raw,
    )

# Estrazione variabili per compatibilità con celle downstream
# pf_rot_std           = results_pipeline["Momentum"]["pf_rot"]
# pf_rot_std_base      = results_pipeline["Momentum"]["pf_rot_base"]
# regime               = results_pipeline["Momentum"]["regime"]
# summary_df_std       = results_pipeline["Momentum"]["summary_df"]
# sel_tickers_std      = results_pipeline["Momentum"]["sel_tickers"]
# sel_tickers_std_base = results_pipeline["Momentum"]["sel_tickers_base"]

# pf_rot_std_v2           = results_pipeline["Multifactor"]["pf_rot"]
# pf_rot_std_base_v2      = results_pipeline["Multifactor"]["pf_rot_base"]
# summary_df_std_v2       = results_pipeline["Multifactor"]["summary_df"]
# sel_tickers_std_v2      = results_pipeline["Multifactor"]["sel_tickers"]
# sel_tickers_std_base_v2 = results_pipeline["Multifactor"]["sel_tickers_base"]

# # Verifica colonna universe
# for _lbl, _sel in [("Momentum", sel_tickers_std_base), ("Multifactor", sel_tickers_std_base_v2)]:
#     assert 'universe' in _sel.columns, f"Colonna 'universe' mancante in sel_tickers_base ({_lbl})"
#     assert _sel['universe'].apply(len).min() > 0
#     print(f"sel_tickers_base [{_lbl}]: {len(_sel)} righe, universe OK")


In [ ]:
# Estrazione variabili per compatibilità con celle downstream
# pf_rot_std           = results_pipeline["Momentum"]["pf_rot"]
# pf_rot_std_base      = results_pipeline["Momentum"]["pf_rot_base"]
# regime               = results_pipeline["Momentum"]["regime"]
# summary_df_std       = results_pipeline["Momentum"]["summary_df"]
# sel_tickers_std      = results_pipeline["Momentum"]["sel_tickers"]
# sel_tickers_std_base = results_pipeline["Momentum"]["sel_tickers_base"]

# pf_rot_std_v2           = results_pipeline["Multifactor"]["pf_rot"]
# pf_rot_std_base_v2      = results_pipeline["Multifactor"]["pf_rot_base"]
# summary_df_std_v2       = results_pipeline["Multifactor"]["summary_df"]
# sel_tickers_std_v2      = results_pipeline["Multifactor"]["sel_tickers"]
# sel_tickers_std_base_v2 = results_pipeline["Multifactor"]["sel_tickers_base"]

# # Verifica colonna universe
# for _lbl, _sel in [("Momentum", sel_tickers_std_base), ("Multifactor", sel_tickers_std_base_v2)]:
#     assert 'universe' in _sel.columns, f"Colonna 'universe' mancante in sel_tickers_base ({_lbl})"
#     assert _sel['universe'].apply(len).min() > 0
#     print(f"sel_tickers_base [{_lbl}]: {len(_sel)} righe, universe OK")


## §5 — Run WFO

### §5a — WFO Standard

Usa la `reduced_grid` prodotta dalla stability analysis.

### §5b — WFO Clusterizzata

**Nota**: usa `build_cluster_grids` internamente — la `reduced_grid` della stability
non viene applicata. I flag binari variano liberamente per cluster
(debito metodologico, vedere CLAUDE.md).

In [ ]:
# Cluster in revisione, path disabilitato
# (run_wfo_pipeline_legacy_cluster richiede build_cluster_grids_legacy_cluster
#  e walk_forward_rotational_legacy_cluster — funzionali ma non integrati
#  nel loop unificato Momentum/Multifactor. Riabilitare quando il refactor
#  Cluster sarà completato.)

# use_clustering     = True
# adaptive_k         = True
# adaptive_k_method  = 'hybrid'
# max_clusters       = 5
# lookback_days      = 504
# n_top_min          = 2
# save_plots         = True
#
# results_cluster = run_wfo_pipeline_legacy_cluster(
#     stocks_data_raw    = stocks_data_raw,
#     stocks_data        = stocks_data,
#     benchmark_data     = benchmark_data,
#     benchmark_data_raw = benchmark_data_raw,
#     tickers            = tickers,
#     risk_off_data      = risk_off_data,
#     ratio              = ratio,
#     metric             = metric,
#     start_date         = pipeline_start_date,
#     end_date           = end_date,
#     cores              = cores,
#     verbose            = verbose,
#     force_next_year_params = force_next_year_params,
#     use_clustering     = use_clustering,
#     adaptive_k         = adaptive_k,
#     adaptive_k_method  = adaptive_k_method,
#     n_clusters         = max_clusters,
#     lookback_days      = lookback_days,
#     n_top_min          = n_top_min,
#     portfolio_title    = portfolio_title,
#     benchmark_title    = benchmark_title,
#     init_cash          = init_cash,
#     risk_on_off        = True,
#     plot               = True,
#     save_plots         = save_plots,
#     plots_dir          = plots_dir,
#     profile            = profile,
#     asset_type         = portfolio.get('asset_type', 'stock'),
#     wfo_audit_path_base = str(reports_dir / f"{portfolio_title}_{year}"),
# )

pf_rot_cluster = None
pf_rot_cluster_base = None
regime_cluster = None
summary_df_cluster = None
sel_tickers_cluster = None
sel_tickers_cluster_base = None
results_cluster = None


### §5c — Confronto WFO

Confronto metriche di performance Standard vs Clusterizzata.
Il confronto dei verdetti OFC (S1-S4) è in §6.

In [ ]:
metrics_df = compare_wfo_pipelines(
    results         = results_pipeline,
    portfolio_title = portfolio_title,
    benchmark_title = benchmark_title,
    plot_radar      = True,
    save_plots      = True,
    plots_dir       = plots_dir,
)
display(metrics_df)


In [ ]:
def quick_sanity_check(pf_rot, pf_rot_base=None, label="Portfolio", min_flags_to_fail=2):
    """
    Controllo qualitativo veloce su un Portfolio VBT, prima di spendere
    tempo su OFC/MC. Usa solo pf.stats(), nessun bootstrap aggiuntivo.
    Segnala soglie d'allarme empiriche — non sono verità statistica,
    solo euristiche per decidere se vale la pena procedere oltre.

    Returns
    -------
    dict con:
        'proceed' : bool  — True se consigliato procedere a OFC/MC
        'flags'   : list[str] — segnali d'allarme raccolti
        'n_flags' : int

    Verdetto: 'proceed' = False se il numero di segnali d'allarme
    raggiunge o supera min_flags_to_fail (default 2). Soglia empirica,
    non statisticamente derivata — pensata come euristica di pre-filtro,
    non come sostituto di OFC/MC.
    """
    print(f"\n{'='*60}")
    print(f"  Quick Sanity Check — {label}")
    print(f"{'='*60}")

    flags = []

    def _check(pf, sublabel):
        if pf is None:
            print(f"  [{sublabel}] non disponibile, skip")
            return
        s = pf.stats()

        total_trades = s.get('Total Trades', float('nan'))
        win_rate     = s.get('Win Rate [%]', float('nan'))
        profit_factor= s.get('Profit Factor', float('nan'))
        expectancy   = s.get('Expectancy', float('nan'))
        worst_trade  = s.get('Worst Trade [%]', float('nan'))
        best_trade   = s.get('Best Trade [%]', float('nan'))
        sharpe       = s.get('Sharpe Ratio', float('nan'))
        max_dd       = s.get('Max Drawdown [%]', float('nan'))

        print(f"\n  --- {sublabel} ---")
        print(f"  Total Trades     : {total_trades}")
        print(f"  Win Rate [%]     : {win_rate:.2f}")
        print(f"  Profit Factor    : {profit_factor:.3f}")
        print(f"  Expectancy       : {expectancy:.4f}")
        print(f"  Sharpe Ratio     : {sharpe:.3f}")
        print(f"  Max Drawdown [%] : {max_dd:.2f}")
        print(f"  Best/Worst Trade : {best_trade:.2f}% / {worst_trade:.2f}%")

        if profit_factor < 1.0:
            flags.append(f"[{sublabel}] Profit Factor < 1.0 ({profit_factor:.3f}) — perdite superano i guadagni in valore assoluto")
        if expectancy < 0:
            flags.append(f"[{sublabel}] Expectancy negativa ({expectancy:.4f}) — il sistema perde in media per trade")
        if sharpe < 0.3:
            flags.append(f"[{sublabel}] Sharpe Ratio basso ({sharpe:.3f}) — rendimento risk-adjusted debole")
        if best_trade and abs(worst_trade) > 3 * abs(best_trade):
            flags.append(f"[{sublabel}] Worst Trade ({worst_trade:.2f}%) >> Best Trade ({best_trade:.2f}%) — possibile outlier di coda pesante")
        if not pd.isna(total_trades) and not pd.isna(win_rate):
            if win_rate > 55 and profit_factor < 1.05:
                flags.append(f"[{sublabel}] Win Rate alto ({win_rate:.1f}%) ma Profit Factor vicino/sotto 1 — pattern 'tanti piccoli vincenti, poche grandi perdite'")

    _check(pf_rot,      "Risk ON/OFF")
    _check(pf_rot_base, "Base")

    proceed = len(flags) < min_flags_to_fail

    print(f"\n{'-'*60}")
    if flags:
        print(f"  ⚠ {len(flags)} segnali d'allarme:")
        for f in flags:
            print(f"    - {f}")
    else:
        print(f"  ✓ Nessun segnale d'allarme evidente.")

    print(f"\n  VERDETTO: {'✓ PROCEDI con OFC/MC' if proceed else '✗ SCONSIGLIATO procedere — rivedere turnover/griglia prima'}")
    print(f"  (soglia: {min_flags_to_fail}+ segnali = sconsigliato; trovati: {len(flags)})")
    print(f"{'='*60}\n")

    return {
        'proceed': proceed,
        'flags':   flags,
        'n_flags': len(flags),
    }

    
# check_v1 = quick_sanity_check(pf_rot_std, pf_rot_std_base, label="Alpha World — Momentum")
# check_v2 = quick_sanity_check(pf_rot_std_v2, pf_rot_std_base_v2, label="Alpha World — Multifactor")

# # Riusabile programmaticamente, es. per skip condizionale:
# if not check_v2['proceed']:
#     print("Salto OFC/MC per v2 (troppi segnali d'allarme)")

In [ ]:
# ── §6 — OFC + MC per ciascun engine ──────────────────────────────────────────
# param_grid: list[dict] da build_wfo_grid (n_trials = len(list), no cartesiano)
# results_pipeline augmentato con chiavi sanity/ofc/mc per ogni engine.

wfo_runs = {
    "Momentum": dict(
        summary_df       = results_pipeline["Momentum"]["summary_df"],
        param_grid       = build_wfo_grid(engine="Momentum", profile=profile,
                               asset_type=portfolio.get('asset_type', 'stock')),
        pf_rot           = results_pipeline["Momentum"]["pf_rot"],
        pf_rot_base      = results_pipeline["Momentum"]["pf_rot_base"],
        sel_tickers      = results_pipeline["Momentum"]["sel_tickers"],
        sel_tickers_base = results_pipeline["Momentum"]["sel_tickers_base"],
        regime           = results_pipeline["Momentum"]["regime"],
    ),
    "Multifactor": dict(
        summary_df       = results_pipeline["Multifactor"]["summary_df"],
        param_grid       = build_wfo_grid(engine="Multifactor", profile=profile,
                               asset_type=portfolio.get('asset_type', 'stock')),
        pf_rot           = results_pipeline["Multifactor"]["pf_rot"],
        pf_rot_base      = results_pipeline["Multifactor"]["pf_rot_base"],
        sel_tickers      = results_pipeline["Multifactor"]["sel_tickers"],
        sel_tickers_base = results_pipeline["Multifactor"]["sel_tickers_base"],
        regime           = None,
    ),
}

for name, cfg in wfo_runs.items():
    print(f"\n{'#'*60}\n  {name}\n{'#'*60}")

    # 1) Sanity check
    check = quick_sanity_check(cfg['pf_rot'], cfg['pf_rot_base'], label=name)

    if not check['proceed']:
        print(f"  [{name}] SKIP OFC/MC — sanity check fallito ({check['n_flags']} flag)")
        results_pipeline[name].update(dict(sanity=check, ofc=None, mc=None))
        continue

    # 2) OFC — param_grid è list[dict]; n_trials = len(list) calcolato internamente
    ofc_passed, ofc_report = overfitting_check_rotational(
        wfo_summary      = cfg['summary_df'],
        stocks_data      = stocks_data,
        benchmark_data   = benchmark_data,
        param_grid       = cfg['param_grid'],
        profile          = profile,
        benchmark_prices = benchmark_data_raw,
        seed             = 42,
        verbose          = True,
    )

    # 3) MC (solo se OFC passa)
    mc_out = None
    if ofc_passed:
        plots_dir_run = Path(plots_dir) / name
        plots_dir_run.mkdir(parents=True, exist_ok=True)
        ci_results, ci_summary_df, skill_results, skill_summary_df = run_all_mc_methods_rotational(
            pf_rot                = cfg['pf_rot'],
            pf_rot_base           = cfg['pf_rot_base'],
            regime                = cfg['regime'],
            sel_tickers           = cfg['sel_tickers'],
            sel_tickers_base      = cfg['sel_tickers_base'],
            stocks_data           = stocks_data,
            benchmark_data        = benchmark_data,
            tickers_master        = tickers,
            init_cash             = init_cash,
            n_simulations         = 1000,
            seed                  = 42,
            block_size            = 10,
            vol_window            = 60,
            n_vol_quantiles       = 3,
            show_method_plots     = True,
            show_method_summaries = True,
            save_plots            = True,
            plots_dir             = plots_dir_run,
        )
        mc_out = dict(ci_results=ci_results, ci_summary_df=ci_summary_df,
                      skill_results=skill_results, skill_summary_df=skill_summary_df)
    else:
        print(f"  [{name}] SKIP MC — OFC non promosso")

    results_pipeline[name].update(dict(
        sanity = check,
        ofc    = dict(passed=ofc_passed, report=ofc_report),
        mc     = mc_out,
    ))

# ── Riepilogo finale ──
print(f"\n{'='*60}\n  RIEPILOGO\n{'='*60}")
for name, r in results_pipeline.items():
    sanity_ok = r.get('sanity', {}).get('proceed')
    ofc_ok    = r.get('ofc', {}).get('passed') if r.get('ofc') else None
    mc_done   = r.get('mc') is not None
    print(f"  {name:12s} | sanity={sanity_ok} | ofc={ofc_ok} | mc_eseguito={mc_done}")


In [ ]:
# for name in ["Momentum", "Multifactor"]:
#     signals = results_pipeline[name]['ofc']['report']['signals']
#     print(f"\n{'='*60}\n  {name} — dettaglio segnali OFC\n{'='*60}")
#     for sig_name, sig_data in signals.items():
#         print(f"  {sig_name}: {sig_data}")

## §6 — Overfitting Check

Valuta 4 segnali di overfitting (S1 plateau, S2 coerenza flag,
S3 cross-sectional skill, S4 DSR). Verdetti separati per Standard e Cluster.
Vedi CLAUDE.md "Interpretation framework: Reshuffle vs S3".

In [ ]:
# import json
# import itertools
# from pathlib import Path

# # --- OFC Standard ---
# ofc_passed_std, ofc_report_std = overfitting_check_rotational(
#     wfo_summary      = summary_df_std,
#     stocks_data      = stocks_data,
#     benchmark_data   = benchmark_data,
#     param_grid       = reduced_grid,
#     profile          = profile,
#     n_total_trials   = n_full_trials,     # penalizzazione conservativa: full grid
#     stability_report = stability_report,
#     seed             = 42,
#     verbose          = True,
# )

# # Salva ofc_report come JSON (audit trail; caricabile in §10 senza re-run)
# ofc_report_path = reports_dir / f"{portfolio_title}_{year}_ofc_std.json"
# Path(ofc_report_path).parent.mkdir(parents=True, exist_ok=True)
# with open(ofc_report_path, "w") as f:
#     json.dump(ofc_report_std, f, default=str, indent=2)
# print(f"OFC report std salvato: {ofc_report_path}")
# print(f"OFC Standard — promoted={ofc_passed_std}")

In [ ]:
# # --- OFC v2 ---
# ofc_passed_std_v2, ofc_report_std_v2 = overfitting_check_rotational(
#     wfo_summary      = summary_df_std_v2,
#     stocks_data      = stocks_data,
#     benchmark_data   = benchmark_data,
#     param_grid       = reduced_grid_v2,
#     profile          = profile,
#     n_total_trials   = n_full_trials_v2,
#     stability_report = stability_report_v2,
#     seed             = 42,
#     verbose          = True,
# )
# ofc_report_path_v2 = reports_dir / f"{portfolio_title}_{year}_ofc_std_v2.json"
# Path(ofc_report_path_v2).parent.mkdir(parents=True, exist_ok=True)
# with open(ofc_report_path_v2, "w") as f:
#     json.dump(ofc_report_std_v2, f, default=str, indent=2)
# print(f"OFC report std v2 salvato: {ofc_report_path_v2}")
# print(f"OFC Standard v2 — promoted={ofc_passed_std_v2}")

In [ ]:
# # --- OFC Cluster (solo se WFO Clusterizzata è stata eseguita) ---
# if 'results_cluster' in locals() and results_cluster is not None:
#     ofc_passed_cluster, ofc_report_cluster = overfitting_check_rotational(
#         wfo_summary      = summary_df_cluster,
#         stocks_data      = stocks_data,
#         benchmark_data   = benchmark_data,
#         param_grid       = full_grid,       # cluster usa full_grid (no stability)
#         profile          = profile,
#         n_total_trials   = n_full_trials,
#         seed             = 42,
#         verbose          = True,
#     )
#     ofc_report_cluster_path = reports_dir / f"{portfolio_title}_{year}_ofc_cluster.json"
#     Path(ofc_report_cluster_path).parent.mkdir(parents=True, exist_ok=True)
#     with open(ofc_report_cluster_path, "w") as f:
#         json.dump(ofc_report_cluster, f, default=str, indent=2)
#     print(f"OFC report cluster salvato: {ofc_report_cluster_path}")
#     print(f"OFC Cluster — promoted={ofc_passed_cluster}")

#     # Tabella comparativa OFC Standard vs Cluster
#     print("\n=== Confronto OFC Standard vs Cluster ===")
#     for sig in ['s1_plateau', 's2_coherence', 's3_random_selection', 's4_dsr']:
#         v_std = ofc_report_std.get(sig, {}).get('passed', '?')
#         v_clu = ofc_report_cluster.get(sig, {}).get('passed', '?')
#         print(f"  {sig:25s}  STD={'PASS' if v_std else 'FAIL'}  CLU={'PASS' if v_clu else 'FAIL'}")
#     print(f"  {'Overall':25s}  STD={'PASS' if ofc_passed_std else 'FAIL'}  "
#           f"CLU={'PASS' if ofc_passed_cluster else 'FAIL'}")
# else:
#     ofc_passed_cluster = None
#     ofc_report_cluster = None
#     print("WFO Clusterizzata non eseguita — OFC Cluster non disponibile.")

## §7 — MC (Monte Carlo Validation)

In [ ]:

# from pathlib import Path

# # Sub-directory per i plot dei due path (evita sovrascrittura)
# plots_dir_std     = Path(plots_dir) / 'std'
# plots_dir_cluster = Path(plots_dir) / 'cluster'
# plots_dir_std_v2  = Path(plots_dir) / 'std_v2'

# plots_dir_std.mkdir(parents=True, exist_ok=True)
# plots_dir_cluster.mkdir(parents=True, exist_ok=True)
# plots_dir_std_v2.mkdir(parents=True, exist_ok=True)


# # Run completa (n=1000) con grafici — Path Standard v1
# ci_results, ci_summary_df, skill_results, skill_summary_df = run_all_mc_methods_rotational(
#     pf_rot              = pf_rot_std,
#     pf_rot_base         = pf_rot_std_base,
#     regime              = regime,
#     sel_tickers         = sel_tickers_std,
#     sel_tickers_base    = sel_tickers_std_base,
#     stocks_data         = stocks_data,
#     benchmark_data      = benchmark_data,
#     tickers_master      = tickers,
#     init_cash           = init_cash,
#     n_simulations       = 1000,
#     seed                = 42,
#     block_size          = 10,
#     vol_window          = 60,
#     n_vol_quantiles     = 3,
#     show_method_plots   = True,
#     show_method_summaries = True,
#     save_plots            = True,
#     plots_dir             = plots_dir_std,   # ← sub-dir Standard
# )

# # Run completa (n=1000) con grafici — Path Standard v2
# ci_results_v2, ci_summary_df_v2, skill_results_v2, skill_summary_df_v2 = run_all_mc_methods_rotational(
#     pf_rot                = pf_rot_std_v2,
#     pf_rot_base           = pf_rot_std_base_v2,
#     regime                = None,   # results_std_v2['regime'] è sempre None nel path v2/Path B
#     sel_tickers            = sel_tickers_std_v2,
#     sel_tickers_base       = sel_tickers_std_base_v2,
#     stocks_data            = stocks_data,
#     benchmark_data          = benchmark_data,
#     tickers_master          = tickers,
#     init_cash               = init_cash,
#     n_simulations           = 1000,
#     seed                    = 42,
#     block_size              = 10,
#     vol_window              = 60,
#     n_vol_quantiles         = 3,
#     show_method_plots       = True,
#     show_method_summaries   = True,
#     save_plots              = True,
#     plots_dir               = plots_dir_std_v2,
# )

# # Run completa (n=1000) con grafici — Path Cluster
# if pf_rot_cluster is not None and pf_rot_cluster_base is not None:
#     ci_results_cluster, ci_summary_df_cluster, skill_results_cluster, skill_summary_df_cluster = run_all_mc_methods_rotational(
#         pf_rot              = pf_rot_cluster,
#         pf_rot_base         = pf_rot_cluster_base,
#         regime              = regime_cluster,
#         sel_tickers         = sel_tickers_cluster,
#         sel_tickers_base    = sel_tickers_cluster_base,
#         stocks_data         = stocks_data,
#         benchmark_data      = benchmark_data,
#         tickers_master      = tickers,
#         init_cash           = init_cash,
#         n_simulations       = 1000,
#         seed                = 42,
#         block_size          = 10,
#         vol_window          = 60,
#         n_vol_quantiles     = 3,
#         show_method_plots   = True,
#         show_method_summaries = True,
#         save_plots            = True,
#         plots_dir             = plots_dir_cluster,   # ← sub-dir Cluster
#     )
# else:
#     ci_results_cluster        = None
#     ci_summary_df_cluster     = None
#     skill_results_cluster     = None
#     skill_summary_df_cluster  = None

In [ ]:
# display(ci_summary_df)
# display(skill_summary_df)
# display(ci_summary_df_cluster)
# display(skill_summary_df_cluster)

## §8 — Decisione finale + Scheda PTF

Aggrega tutti i segnali in una tabella decisionale per path.
L'ultima riga "User decision" è compilata a mano dal modellista.

In [ ]:
# STEP 8 — Decisione finale + Scheda PTF + Relazione Tecnica

_report = generate_final_report(
    results_pipeline           = results_pipeline,
    portfolio_title            = portfolio_title,
    year                       = year,
    profile                    = profile,
    benchmark_title            = benchmark_title,
    tickers                    = tickers,
    pipeline_start_date        = pipeline_start_date,
    wfo_file_save              = wfo_file_save,
    survivorship_bias_universe = survivorship_bias_universe,
    reports_dir                = reports_dir,
    plots_dir                  = plots_dir,
    ratio                      = ratio,
    metric                     = metric,
)


In [ ]:
# # nel kernel Jupyter stesso, non nel terminale bash
# import os
# print(os.path.getmtime('/tmp/relazione_llm_debug.md'))
# import time
# print(time.time())
# print(os.uname())

In [ ]:
# §8 — Salvataggio WFO per il RUNTIME (path scelto dall'architetto)
#
# Compilare `path_scelto` a mano DOPO aver letto la relazione tecnica PDF.
# "STANDARD" → usa summary_df_std (Momentum engine, unified pipeline)
# "CLUSTER"  → usa summary_df_cluster (disabilitato — richiede §5b abilitato)

path_scelto = "STANDARD"   # "STANDARD" oppure "CLUSTER" — compilare a mano dopo lettura PDF

_path_scelto = str(path_scelto).strip().upper()
if _path_scelto not in ("STANDARD", "CLUSTER"):
    raise ValueError(
        f"path_scelto non valido: '{path_scelto}'. Valori attesi: 'STANDARD' o 'CLUSTER'."
    )

if _path_scelto == "CLUSTER":
    if results_cluster is None:
        raise RuntimeError(
            "path_scelto='CLUSTER' ma results_cluster is None: il path Cluster non e' "
            "stato eseguito (§5b — WFO Clusterizzata). Esegui il WFO Cluster prima di "
            "salvare, oppure imposta path_scelto='STANDARD'."
        )
    _summary_df_runtime = summary_df_cluster
    _param_grid_runtime = full_grid       # Cluster usa full_grid (legacy dict)
else:
    _summary_df_runtime = summary_df_std
    # Standard usa build_wfo_grid (list[dict]); se wfo_runs è disponibile, riusa da lì
    _param_grid_runtime = wfo_runs["Momentum"]["param_grid"] if 'wfo_runs' in dir() else \
        build_wfo_grid(engine="Momentum", profile=profile,
                       asset_type=portfolio.get('asset_type', 'stock'))

save_rotational_wfo_summary(
    summary_df             = _summary_df_runtime,
    start_date             = start_date,
    end_date               = end_date,
    file_path              = wfo_file_save,
    param_grid             = _param_grid_runtime,
    metric                 = metric,
    ratio                  = ratio,
    force_next_year_params = force_next_year_params,
    extra_meta             = {"deployed_path": _path_scelto},
)
print(f"[§8] Path scelto per il RUNTIME: {_path_scelto}")
print(f"[§8] WFO summary salvato su (letto da r_run_portfolio): {wfo_file_save}")


## §9 — Performance

In [ ]:
# Ci sono 4 portafogli disponibili:
# pf_rot_std_base        (WFO non clusterizzata)
# pf_rot_std             (WFO non clusterizzata, Risk ON/OFF)
# pf_rot_cluster_base    (WFO clusterizzata)
# pf_rot_cluster         (WFO clusterizzata, Risk ON/OFF)

# E per ognuno di essi 2 possibili confronto con altrettanti benchmark
# Benchmark esterno: definito dal portafoglio (benchmark_data=benchmark_data)
# Benchmark interno: B&H intero universo  (benchmark_data=None)

# Schema decisionale rapido                                                                                                      
                                                                                                                             
# Universo piccolo/omogeneo?
#   ├─ Sì → std_base  (o std se vuoi protezione drawdown)                                                                        
#   └─ No (grande/eterogeneo) → cluster_base  (o cluster se vuoi regime switching)                                               
                                                                                                                             
# Vuoi ridurre il max drawdown?                                                                                                  
#   └─ Sì → aggiungi Risk ON/OFF (std → std_on, cluster → cluster_on)                                                            
                                                                                                                             
# Periodo OOS corto (<3 anni)?
#   └─ Preferisci la versione _base: meno parametri, più robusta                                                                 
                                                                                                                             
# ---
# In pratica, pf_rot_cluster è il più potente ma anche il più fragile se i dati sono pochi. pf_rot_std_base è il più stabile e   
# interpretabile. Gli altri due stanno nel mezzo.                                                                                

pf = pf_rot_cluster
sel_tickers = sel_tickers_cluster

out = generate_rotational_portfolio_performance(
    pf=pf,
    portfolio_title=portfolio_title,
    sel_tickers=sel_tickers,
    benchmark=benchmark_title,
    benchmark_data=benchmark_data,   # prezzi close
    alpha_analysis=True,
    show_plots=True,
    plot_start_date='2026-05-11'
)

In [ ]:
# Engine Sanity Check
health_df, ticker_df, selection_log, details = build_engine_health_check(
    pf_rot_std,
    sel_tickers_std,
    prices    = stocks_data,
    start_date = start_date,
    end_date   = end_date,
    include_prev = True,
)
my_display(health_df, "Health check motore rotazionale")
my_display(selection_log, "Audit selezioni")

## §10 — Load WFO Results

Riprende un'analisi precedente senza rieseguire WFO, stability e OFC.
Richiede che §4 abbia salvato `stability_report.csv` e §6 abbia salvato `ofc_report.json`.

In [ ]:
# Carica wfo_summary, stability_report e ofc_report da run precedente
import json
import os

summary_df       = load_wfo_summary(wfo_file_save)

stability_report_path   = reports_dir / f"{portfolio_title}_{year}_stability.csv"
ofc_report_std_path     = reports_dir / f"{portfolio_title}_{year}_ofc_std.json"

if os.path.exists(stability_report_path):
    stability_report = pd.read_csv(stability_report_path)
    print(f"Stability report caricato: {stability_report_path}")
else:
    stability_report = None
    print(f"[WARN] stability report non trovato: {stability_report_path}")

if os.path.exists(ofc_report_std_path):
    with open(ofc_report_std_path) as f:
        ofc_report_std = json.load(f)
    print(f"OFC report std caricato: {ofc_report_std_path}")
else:
    ofc_report_std = None
    print(f"[WARN] OFC report std non trovato: {ofc_report_std_path}")

display(summary_df)

In [ ]:
# Ricostruisce pf_rot e sel_tickers dalla wfo_summary caricata (§10)
pf_rot, pf_bh, sel_tickers = build_portfolio_from_wfo_summary_legacy_cluster(
    summary_df     = summary_df,
    stocks_data    = stocks_data,
    benchmark_data = benchmark_data,
    start_date     = start_date,
    end_date       = end_date,
    benchmark_title = benchmark_title,
    portfolio_name  = portfolio_title,
    init_cash       = init_cash,
    plot            = True,
    show_report     = True,
)


In [ ]:
# Ricostruisce pf_rot e sel_tickers dal cluster wfo_summary (§10)
pf_rot, pf_bh, sel_tickers = build_portfolio_from_wfo_summary_legacy_cluster(
    summary_df     = summary_df_cluster,
    stocks_data    = stocks_data,
    benchmark_data = benchmark_data,
    start_date     = start_date,
    end_date       = end_date,
    benchmark_title = benchmark_title,
    portfolio_name  = portfolio_title,
    init_cash       = init_cash,
    plot            = True,
    show_report     = True,
)


In [ ]:
# Dopo il load e' possibile eseguire §8 (Decisione finale) ricalcolando
# le variabili derivate da ofc_report_std e skill_results se disponibili.
# Oppure ri-eseguire §7 (MC) senza rieseguire il WFO.
print("Load completato. Variabili disponibili:")
print(f"  summary_df:       {summary_df.shape}")
print(f"  pf_rot:           {type(pf_rot).__name__}")
print(f"  sel_tickers:      {sel_tickers.shape}")
print(f"  stability_report: {'OK' if stability_report is not None else 'non disponibile'}")
print(f"  ofc_report_std:   {'OK' if ofc_report_std is not None else 'non disponibile'}")

## Snapshot

In [ ]:
# # Snapshot pre-fix MC — portfolio_germany_plan
# import shutil, json
# from pathlib import Path
# from datetime import datetime

# snap_dir = Path(f"../dev/snapshots/pre_mc_fix/{portfolio_title}").resolve()
# snap_dir.mkdir(parents=True, exist_ok=True)

# # 1. Re-cattura output testuale di print_final_decision
# import io, contextlib
# buf = io.StringIO()
# with contextlib.redirect_stdout(buf):
#     print_final_decision(
#         portfolio_title    = portfolio_title,
#         year               = year,
#         profile            = profile,
#         ofc_report_std     = ofc_report_std,
#         ofc_report_cluster = ofc_report_cluster,
#         mc_skill           = skill_results,
#         mc_ci              = ci_summary_df,
#         skill_profile      = skill_profile,
#     )
# (snap_dir / "print_final_decision.txt").write_text(buf.getvalue())

# # 2. Copia PTF Card
# ptf_card_src = Path(_card_path).resolve()
# if ptf_card_src.exists():
#     shutil.copy(ptf_card_src, snap_dir / "ptf_card.md")

# # 3. Copia scheda tecnica PDF
# pdf_src = Path(_pdf_path).resolve()
# if pdf_src.exists():
#     shutil.copy(pdf_src, snap_dir / "scheda_tecnica.pdf")

# # 4. Metadati snapshot
# metadata = {
#     "portfolio_title": portfolio_title,
#     "snapshot_date": datetime.now().isoformat(),
#     "branch": "fix/r-mc-cluster-symmetry",
#     "bug_status": "PRESENT (pre-fix baseline)",
#     "files": {
#         "print_final_decision.txt": (snap_dir / "print_final_decision.txt").exists(),
#         "ptf_card.md": (snap_dir / "ptf_card.md").exists(),
#         "scheda_tecnica.pdf": (snap_dir / "scheda_tecnica.pdf").exists(),
#     },
# }
# (snap_dir / "metadata.json").write_text(json.dumps(metadata, indent=2))

# print(f"Snapshot salvato: {snap_dir}")
# for k, v in metadata["files"].items():
#     print(f"  {k}: {'OK' if v else 'MISSING'}")